# Session affinity for httpbin Deployments

This notebook creates one shared httpbin Tool and three independent Deployments to compare every affinity mode. Affinity configuration cannot be changed with `deployment update`, so each mode needs its own Deployment.

| Mode | When the target instance is unavailable | Instance ownership |
| --- | --- | --- |
| `BEST_EFFORT` | Another instance may be selected | Shared |
| `STRICT` | The request fails; it does not migrate | Shared |
| `EXCLUSIVE` | It does not migrate | One instance is dedicated to each affinity ID |

Requests and responses use `X-Httpbin-Affinity`. The example observes routing through response headers and HTTP status. It does not expose real hostnames or use validation scripts. Replace `AGR_ROLE_ARN` with a CAM role ARN that lets AGR pull the target CCR image.

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-affinity-your-name
%env BEST_EFFORT_DEPLOYMENT_NAME=httpbin-best-effort-your-name
%env STRICT_DEPLOYMENT_NAME=httpbin-strict-your-name
%env EXCLUSIVE_DEPLOYMENT_NAME=httpbin-exclusive-your-name
!agr status

## 1. Create the shared Tool

Replace `your-name` in all four names with the same unique suffix.

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. Create three Deployments

Copy `ToolId`. All three Deployments use the same header name and a 30-second `STOP` idle policy so you can observe what happens when a target instance becomes unavailable. `EXCLUSIVE` permits at most three dedicated instances.

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create --region "$AGR_REGION" --deployment-name "$BEST_EFFORT_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":2,"MaxInstanceRequestConcurrency":10}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"BEST_EFFORT","HeaderName":"X-Httpbin-Affinity"}'
!agr deployment create --region "$AGR_REGION" --deployment-name "$STRICT_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":2,"MaxInstanceRequestConcurrency":10}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"STRICT","HeaderName":"X-Httpbin-Affinity"}'
!agr deployment create --region "$AGR_REGION" --deployment-name "$EXCLUSIVE_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":3,"MaxInstanceRequestConcurrency":1}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"EXCLUSIVE","HeaderName":"X-Httpbin-Affinity"}'

## 3. Acquire three Deployment tokens

Copy the three `DeploymentId` values in create order. Acquire a token for each Deployment, then copy each response's `Data.Response.Response.Token` into the following cell. Tokens cannot be shared across Deployments.

In [ ]:
%env BEST_EFFORT_DEPLOYMENT_ID=dpl-replace-me
%env STRICT_DEPLOYMENT_ID=dpl-replace-me
%env EXCLUSIVE_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$BEST_EFFORT_DEPLOYMENT_ID'"}' --output json
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$STRICT_DEPLOYMENT_ID'"}' --output json
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$EXCLUSIVE_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env BEST_EFFORT_TOKEN=dpt-replace-me
%env STRICT_TOKEN=dpt-replace-me
%env EXCLUSIVE_TOKEN=dpt-replace-me

## 4. `BEST_EFFORT`: prefer reuse, allow migration

The first request omits the affinity header. Copy the response's `X-Httpbin-Affinity` value into the second cell. Returning that value asks the Deployment to prefer the previous instance. If you idle for at least 30 seconds so that instance stops, returning the same ID can still select another instance and continue.

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $BEST_EFFORT_TOKEN" "https://8080-$BEST_EFFORT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env BEST_EFFORT_AFFINITY_ID=replace-with-response-header
!curl --include --silent --show-error --header "X-Access-Token: $BEST_EFFORT_TOKEN" --header "X-Httpbin-Affinity: $BEST_EFFORT_AFFINITY_ID" "https://8080-$BEST_EFFORT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 5. `STRICT`: require reuse, forbid migration

Acquire and copy the response affinity ID, then return it in a request. Next, idle for at least 30 seconds so the target instance stops. Returning the same ID again should fail under `STRICT` instead of selecting a new instance. `--include` keeps the HTTP status and error response visible.

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $STRICT_TOKEN" "https://8080-$STRICT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env STRICT_AFFINITY_ID=replace-with-response-header
!curl --include --silent --show-error --header "X-Access-Token: $STRICT_TOKEN" --header "X-Httpbin-Affinity: $STRICT_AFFINITY_ID" "https://8080-$STRICT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 6. `EXCLUSIVE`: one dedicated instance per affinity ID

Send two requests without affinity headers and copy the two response IDs. The two IDs own separate, non-shared, non-migrating instances; later requests must return the corresponding ID. The instance ceiling therefore also limits simultaneous exclusive sessions.

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env EXCLUSIVE_AFFINITY_ID_A=replace-with-first-response-header
%env EXCLUSIVE_AFFINITY_ID_B=replace-with-second-response-header
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" --header "X-Httpbin-Affinity: $EXCLUSIVE_AFFINITY_ID_A" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" --header "X-Httpbin-Affinity: $EXCLUSIVE_AFFINITY_ID_B" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 7. Clean up

Delete all Deployments first. After every deletion finishes, delete the shared Tool.

In [ ]:
!agr deployment delete "$BEST_EFFORT_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment delete "$STRICT_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment delete "$EXCLUSIVE_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

If any instance is not `STOPPED`, copy each ID in turn. In a new cell, run `%env HTTPBIN_INSTANCE_ID=replace-me`, followed by `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`. Delete the shared Tool after all such instances are gone.

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait